In [11]:
import joblib
import pandas as pd
import numpy as np
import sklearn
import xgboost
import flaml
from flaml import AutoML

In [12]:
# Load processed splits or dataframes
X_train = joblib.load("../models/X_train.pkl")
X_test = joblib.load("../models/X_test.pkl")
y_train = joblib.load("../models/y_train.pkl")
y_test = joblib.load("../models/y_test.pkl")

print(f"X_train shape: {X_train.shape}")
print(f"y_train positive class ratio: {y_train.mean():.4f}")

X_train shape: (226980, 30)
y_train positive class ratio: 0.0017


In [13]:
automl = AutoML()

In [14]:
automl.fit(
    X_train, y_train,                 # full train, no X_tr/X_val
    task="classification", metric="ap",
    estimator_list=["xgboost"],
    eval_method="cv", n_splits=5,
    time_budget=300, seed=42,
    log_file_name="../reports/flaml_xgb_fraud.log"
)

[flaml.automl.logger: 10-02 20:52:24] {2470} INFO - task = classification
[flaml.automl.logger: 10-02 20:52:24] {2481} INFO - Evaluation method: cv
[flaml.automl.logger: 10-02 20:52:24] {2584} INFO - Minimizing error metric: 1-ap
[flaml.automl.logger: 10-02 20:52:24] {2701} INFO - List of ML learners in AutoML Run: ['xgboost']
[flaml.automl.logger: 10-02 20:52:24] {3006} INFO - iteration 0, current learner xgboost
[flaml.automl.logger: 10-02 20:52:25] {3141} INFO - Estimated sufficient time budget=8315s. Estimated necessary time budget=8s.
[flaml.automl.logger: 10-02 20:52:25] {3192} INFO -  at 1.6s,	estimator xgboost's best error=3.1451e-01,	best estimator xgboost's best error=3.1451e-01
[flaml.automl.logger: 10-02 20:52:25] {3006} INFO - iteration 1, current learner xgboost
[flaml.automl.logger: 10-02 20:52:26] {3192} INFO -  at 2.5s,	estimator xgboost's best error=3.1451e-01,	best estimator xgboost's best error=3.1451e-01
[flaml.automl.logger: 10-02 20:52:26] {3006} INFO - iteration

In [15]:
from sklearn.metrics import (
    average_precision_score,
    precision_recall_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report
)

# 1. Get probability of the positive class (fraud)
y_pred_proba = automl.predict_proba(X_test)[:, 1]

# 2. Evaluate test Average Precision (PR-AUC)
test_ap = average_precision_score(y_test, y_pred_proba)
print(f"Test Average Precision (PR-AUC): {test_ap:.4f}")

# 3. Find optimal decision threshold via Precision-Recall curve (maximizing F1)
precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba)
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
best_threshold_idx = np.argmax(f1_scores)
optimal_threshold = thresholds[best_threshold_idx] if best_threshold_idx < len(thresholds) else 0.5
print(f"Optimal Threshold (by F1): {optimal_threshold:.4f}")

# 4. Apply the optimal threshold to get binary predictions
y_pred_optimal = (y_pred_proba >= optimal_threshold).astype(int)

# 5. Calculate metrics at the custom threshold
acc = accuracy_score(y_test, y_pred_optimal)
prec = precision_score(y_test, y_pred_optimal, zero_division=0)
rec = recall_score(y_test, y_pred_optimal, zero_division=0)
f1 = f1_score(y_test, y_pred_optimal, zero_division=0)

# 6. Print formatted results matching your leaderboard keys
print("\n=== Evaluation Summary ===")
print(f"Model Accuracy: {acc:.4f}")
print(f"Precision:      {prec:.4f}")
print(f"Recall:         {rec:.4f}")
print(f"F1 Score:       {f1:.4f}")
print(f"PR-AUC:         {test_ap:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred_optimal))

Test Average Precision (PR-AUC): 0.8184
Optimal Threshold (by F1): 0.2222

=== Evaluation Summary ===
Model Accuracy: 0.9996
Precision:      0.9605
Recall:         0.7684
F1 Score:       0.8538
PR-AUC:         0.8184

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.96      0.77      0.85        95

    accuracy                           1.00     56746
   macro avg       0.98      0.88      0.93     56746
weighted avg       1.00      1.00      1.00     56746



In [16]:
bundle = {
    "model": automl.model.estimator,        # The plain XGB model instance
    "best_estimator": automl.best_estimator,
    "best_config": automl.best_config,
    "feature_names": list(X_train.columns), # Preserves exact feature column order
    "threshold": float(optimal_threshold),  
    "metric": "ap",
    "seed": 42,
    "versions": {
        "flaml": flaml.__version__,
        "xgboost": xgboost.__version__,
        "sklearn": sklearn.__version__,
    },
}

# Save bundle and full object
joblib.dump(bundle, "../models/flaml_bundle.joblib")
joblib.dump(automl, "../models/flaml_automl_full.joblib")

print("Successfully saved flaml_bundle.joblib to models/")

Successfully saved flaml_bundle.joblib to models/
